# 02 — LLM Chat & Baseline Survey

Test LLM chat integration, parse survey responses, run baseline for sample agents, compare LLM vs real survey answers.

**Covers:** Issue 2 (LLM chat), Issue 3 (baseline survey), Issue 4 (political agents)

##  LLM Chat

The src/abm/io/llm.py file is a wrapper for sending LLM calls. 

In [1]:
import sys, os, random
sys.path.insert(0, os.path.abspath("../src"))

from cag.io.llm import send_chat

system_prompt = "You are a chef who is obsessed wiht healthy eating"
user_prompt = "Hey I amde a cheese and ahm sandwich, what do you think?"

llm_response = send_chat(system_prompt, user_prompt, api_key=None, model="gpt-4o-mini",
              provider="openai", temperature=0.7)

print(llm_response)

A cheese and ham sandwich can be delicious, but there are definitely ways to make it healthier! Here are a few tips:

1. **Choose Whole Grain Bread**: Opt for whole grain or whole wheat bread instead of white bread. It provides more fiber and nutrients.

2. **Lean Ham**: If possible, select lean ham or turkey instead of regular ham to reduce saturated fat and sodium.

3. **Cheese Choice**: Consider using a lower-fat cheese or a cheese with higher protein content, like cottage cheese or part-skim mozzarella.

4. **Add Veggies**: Load your sandwich with fresh vegetables like spinach, lettuce, tomatoes, or cucumbers for added vitamins and minerals.

5. **Healthy Spread**: Instead of mayo or butter, try hummus, mustard, or avocado as a spread for healthy fats.

These adjustments can make your sandwich not only tastier but also more nutritious! Enjoy your meal!


## We want to ask the LLM survey questions



In [2]:
system_prompt = "You are a climate change activist who is obsessed with reducing carbon emissions"
user_prompt = ("What do you think of a polcy to reduce carbon emissions by 50% by 2030?" +
               "A: Strongly Oppose, B: Somewhat Oppose, C: Slightly Oppose, D: Neutral, E: Agree, F: Somewhat Agree, G: Strongly Agree" +
               "Please respond with just the letter corresponding to your answer.")

llm_response = send_chat(system_prompt, user_prompt, api_key=None, model="gpt-4o-mini",
              provider="openai", temperature=0.7)

print("LLM Response: " + llm_response)

from cag.io.llm import parse_letter_response
letter_response = parse_letter_response(llm_response)

print("Parsed Response: " + letter_response)

LLM Response: G
Parsed Response: G


It can parse letters from more verbose responses.

In [3]:
print(parse_letter_response("I think the answer is A"))
print(parse_letter_response("B, because I don't like it"))
print(parse_letter_response("C"))

A
B
C


Throws an error if it cannot parse:

In [4]:
# If it cannot parse it throws a ValueError:
print(parse_letter_response("I think this is a great policy and I support it"))

ValueError: Could not extract a valid letter A-G from response: 'I think this is a great policy and I support it'

## Survey - Baseline

The src/cag/abm/agent.py file has a SurveyedCitizen class that handles taking surveys. 

In [5]:
# Imports
from gabm.abm.attributes.gender import GenderID, GenderMap
from gabm.abm.attributes.ethnicity import EthnicityID
from gabm.abm.attributes.income import IncomeID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.attributes.education import EducationID
from gabm.abm.attributes.region import RegionID
from gabm.abm.attributes.family import FamilyID
from gabm.abm.democracy.election import ElectionID

from cag.io.survey import load
from cag.abm.environment import SurveyedNation
from cag.abm.agent import SurveyedCitizen
from cag.abm.attributes.ethnicity import SurveyEthnicityMap
from cag.abm.attributes.income import SurveyIncomeMap
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.education import SurveyEducationMap
from cag.abm.attributes.region import UKRegionMap
from cag.abm.attributes.family import SurveyFamilyMap
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteID, UKGE2019, UKGE2019VoteMap
from cag.abm.democracy.elections.brexit import BrexitVoteID, Brexit, BrexitVoteMap
from cag.abm.attributes.narratives import (
    NarrativeAttributeID, SelftranscMap, SelfenhMap,
    OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7
)

print("Imports OK")

Imports OK


In [6]:
random.seed(42)
year = 2026

# Elections
UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

# Attribute maps
surveyed_nation = SurveyedNation(
    year=year, place="UK",
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap,
    selfenh_map=SelfenhMap,
    openness_map=OpennessMap,
    conformtrad_map=ConformTradMap,
    sdo_map=SDOMap,
    edo_map=EDOMap,
    rwa_map=RWAMap,
)

# Load survey data
data = load("../data/yougov_survey_data/YouGovProcessedData.csv")
print(f"Loaded {len(data)} survey rows")

Loaded 1086 survey rows


In [7]:
# Create agents from survey rows
agents = []
for i in range(len(data)):
    row = data.iloc[i]
    agent_id = row.get('ID', None)
    age = int(row.get('age', 0))
    year_of_birth = year - age
    gender_id = GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE
    region_id = RegionID(int(row.get('tprofile_GOR', 0)))
    education_id = EducationID(int(row.get('profile_education_level', 0)))
    income_id = IncomeID(int(row.get('tprofile_gross_household', 0)))
    ethnicity_id = EthnicityID(int(row.get('ethnicity_R', 0)))
    family_id = FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT
    ukge2019_vote_id = UKGE2019VoteID(int(row.get('Vote2019R', 0)))
    brexit_vote_id = BrexitVoteID(int(row.get('pastvote_EURef', 0)))
    politics_id = PoliticsID(int(row.get('Political_Left_Right', 0)))
    selftransc_id = rescale_1_6(int(row.get('Selftransc_Val', 0)))
    selfenh_id = rescale_1_6(int(row.get('Selfenh_Values', 0)))
    openness_id = rescale_1_6(int(row.get('Openness', 0)))
    conformtrad_id = rescale_1_6(int(row.get('ConformTrad', 0)))
    sdo_id = rescale_1_7(int(row.get('SDO', 0)))
    edo_id = rescale_1_7(int(row.get('EDO', 0)))
    rwa_id = rescale_1_6(int(row.get('RWA', 0)))

    sc = SurveyedCitizen(
        agent_id=agent_id, environment=surveyed_nation,
        year_of_birth=year_of_birth, gender_id=gender_id,
        opinions=None, region_id=region_id, ethnicity_id=ethnicity_id,
        income_id=income_id, education_id=education_id,
        politics_id=politics_id, family_id=family_id,
        ukge2019_vote_id=ukge2019_vote_id, brexit_vote_id=brexit_vote_id,
        selftransc_id=selftransc_id, selfenh_id=selfenh_id,
        openness_id=openness_id, conformtrad_id=conformtrad_id,
        sdo_id=sdo_id, edo_id=edo_id, rwa_id=rwa_id,
        original_survey_data=data.iloc[i],
    )
    agents.append(sc)

for sc in agents:
    surveyed_nation.agents_active[sc.id] = sc

print(f"Created {len(agents)} agents")

Created 1086 agents


In [8]:
agent_index = 2
agent = agents[agent_index]
# 

from cag.abm.attributes.opinion import ClimatePolicyID
agent_system_prompt = agent.get_system_prompt()
print("System prompt:" + agent_system_prompt)

agent_user_prompt = agent.get_user_prompt(policy_id = ClimatePolicyID.BAN_FOSSIL_FUEL)
print("User prompt:"  + agent_user_prompt)

letter, numeric = agent.administer_survey(ClimatePolicyID.BAN_FOSSIL_FUEL)
print(f"Agent response to BAN_FOSSIL_FUEL: Letter={letter}, Numeric={numeric}")

print(f"Real survey response to BAN_FOSSIL_FUEL: {agent.get_real_survey_response(policy_id = ClimatePolicyID.BAN_FOSSIL_FUEL)}")

System prompt:I am a 57 year old female living in the East Midlands. My ethnicity is white. I have a city & guilds certificate. My gross household income is £30,000 - £34,999 per year. I am a parent. I position myself centre of the political spectrum. I voted to leave in the 2016 EU Referendum.
When it comes to my core values and worldview: I care about the people close to me and have a basic respect for nature, but I do not actively champion global equality or make environmental protection a primary, driving life focus. I am not strongly driven by the need to get ahead of others, impress people, or hold leadership positions where I tell others what to do. I prefer routine and the familiar, showing little interest in taking risks, seeking out new adventures, or coming up with highly original ideas. I place a high value on obedience, showing deep respect for parents and older people. I strongly believe in maintaining traditional ways of thinking, keeping up customs, and always behaving 

# Running baseline from a SurveyedNation



In [9]:
# The method takes a max_agets parameter to limit how many agents it runs on, otherwise it would send thousands of LLM calls. 
max_agents = 5

surveyed_nation.run_baseline(max_agents=max_agents)

Overall baseline accuracy: 26.7% (exact match), 0.717 (ordinal score)
Policy ClimatePolicyID(1) - Exact: 40.0%, Ordinal: 0.900
Policy ClimatePolicyID(2) - Exact: 0.0%, Ordinal: 0.633
Policy ClimatePolicyID(3) - Exact: 20.0%, Ordinal: 0.633
Policy ClimatePolicyID(4) - Exact: 40.0%, Ordinal: 0.900
Policy ClimatePolicyID(5) - Exact: 20.0%, Ordinal: 0.600
Policy ClimatePolicyID(6) - Exact: 40.0%, Ordinal: 0.633


,agent_id,policy_id,llm_letter,llm_numeric,real_response,match,ordinal_score
0,1.0,ClimatePolicyID(1),G,3,2,False,0.833333
1,1.0,ClimatePolicyID(2),G,3,0,False,0.500000
2,1.0,ClimatePolicyID(3),F,2,-3,False,0.166667
3,1.0,ClimatePolicyID(4),G,3,2,False,0.833333
4,1.0,ClimatePolicyID(5),G,3,-3,False,0.000000
5,1.0,ClimatePolicyID(6),G,3,-3,False,0.000000
6,2.0,ClimatePolicyID(1),G,3,2,False,0.833333
7,2.0,ClimatePolicyID(2),G,3,-1,False,0.333333
8,2.0,ClimatePolicyID(3),F,2,-1,False,0.500000
9,2.0,ClimatePolicyID(4),G,3,2,False,0.833333


## Political Agents (Issue 4)

Two fixed-stance political agents generate persuasive messages about climate policies. They do not update their opinions — they are message sources that broadcast to connected citizens.

- **Agent A (Pro-Climate):** Eco-populist, Green Party UK archetype
- **Agent B (Anti-Climate):** Right-wing, Reform Party UK archetype

In [1]:
from cag.abm.agent import PoliticalAgent
from cag.abm.attributes.opinion import ClimatePolicyID

# Create pro-climate and anti-climate political agents (default system prompts)
agent_a = PoliticalAgent("political_agent_a", "pro_climate")
agent_b = PoliticalAgent("political_agent_b", "anti_climate")

# Generate a persuasive message from each agent for the CARBON_TAX policy
policy = ClimatePolicyID.CARBON_TAX

print("=== Pro-Climate Agent (Agent A) ===")
msg_a = agent_a.generate_message(policy)
print(msg_a)

print("\n=== Anti-Climate Agent (Agent B) ===")
msg_b = agent_b.generate_message(policy)
print(msg_b)

=== Pro-Climate Agent (Agent A) ===
Friends, let’s be clear: the climate crisis isn’t just an environmental issue—it’s a crisis of justice, a crisis of inequality. Right now, the fossil fuel giants are raking in record profits while ordinary families are left to shoulder the burden of rising energy costs and a warming planet. It’s time to fight back! 

I wholeheartedly support a carbon tax on fossil fuel sales, but let’s make sure it works for us, the people. By implementing a carbon fee and dividend system, we can hold the polluters accountable while putting money directly back into the pockets of everyday folks. This means lower energy bills and more financial security for families struggling to make ends meet. 

Imagine using that dividend to insulate your home, to invest in renewable energy, or simply to help pay your bills. This is not just about saving the planet; it’s about taking back control from the billionaire class who profit from our struggles. Let’s demand a fairer, green